# 04 – Rule / Keyword Baseline

**Goal.** Build the simplest possible, non-ML "model" that does what the final RAG system must do:
for each of the 54 Starhawk SRS requirements, (1) cite the most relevant NASA clauses and
(2) say whether the requirement is `ACCEPTABLE` or `NEEDS_REVIEW`, with a written reason.

**Why a baseline?** It is the floor. If the RAG/LLM system cannot beat a word list + TF-IDF, it is not
worth the cost. The evaluation step later compares the two.

**Two parts:**

| Part | Method | Produces |
|---|---|---|
| Retrieval | TF-IDF + cosine similarity | top-3 `chunk_id`s per requirement (the citations) |
| Judgment | keyword rules from `requirement_quality_taxonomy.md` | `quality_status` + `issue_codes` + `reason` |

**Inputs**

| File | What it is |
|---|---|
| `data/srs_requirements.csv` | 54 requirements extracted from `data/Starhawk Mission Computer SRS.md` |
| `data/addressable_clauses.csv` | 62 NASA clauses that apply to an SRS (notebooks 01/02/05) |
| `docs/sprint-weeks-1-2/requirement_quality_taxonomy.md` | defect categories A1–A8 → source of the keyword lists |
| `docs/sprint-weeks-1-2/manual_requirement_review.csv` | manual labels for all 54 → used to score the baseline |
| `Sprint4_Team_Starter_DRAFT/calibration_examples_for_teams_v1.csv` | 11 calibration cases from the challenge advisor → second check |

**Output:** `data/baseline_predictions.csv`, one row per requirement, using the label names from
`output_contract_v1.md`: `ACCEPTABLE / NEEDS_REVIEW`, issue codes such as `VAGUE_CRITERION`.

`output_contract_v1.md` says: *"do not turn cosine similarity into verdicts."* So similarity is only used for citations;
the label always comes from a keyword hit, and the `reason` column names that keyword.

## 1. Setup – load the two input tables

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 120)

# Paths are relative to the notebooks/ folder.
reqs    = pd.read_csv("../data/srs_requirements.csv")       # req_id, title, text, section
clauses = pd.read_csv("../data/addressable_clauses.csv")    # chunk_id, swe_id, section, text, source, ...

print("requirements:", reqs.shape)   # expect (54, 4)
print("clauses:     ", clauses.shape) # expect (62, 10)
reqs.head(3)

requirements: (54, 4)
clauses:      (62, 10)


,req_id,title,text,section
0,SFMC-REQ-001,Position Display,The SFMC shall display the spacecraft's current position to the pilot.,3.1 Navigation
1,SFMC-REQ-002,Navigation Update,The SFMC shall update the displayed spacecraft position at least once every 500 milliseconds while the spacecraft is...,3.1 Navigation
2,SFMC-REQ-003,Destination Selection,The SFMC shall allow the pilot to select a navigation destination.,3.1 Navigation


In [2]:
clauses.head(3)

,chunk_id,swe_id,section,text,source,source_url,type,applies_to,interpretation,review_note
0,npr-3.7.3-requirement,SWE-134,3.7.3,"If a project has safety-critical software or mission-critical software, the project manager shall implement the foll...",NPR 7150.2D,https://nodis3.gsfc.nasa.gov/displayDir.cfm?Internal_ID=N_PR_7150_002D_&page_name=Chapter3,requirement,MULTIPLE,"Requires implementation of safety-critical software behaviors, but an SRS can also be checked for whether correspond...",Same safety list as SWEHB 3.2.1.4. Count as one finding.
1,npr-3.10.1-requirement,SWE-147,3.10.1,The project manager shall specify reusability requirements that apply to its software development activities to enab...,NPR 7150.2D,https://nodis3.gsfc.nasa.gov/displayDir.cfm?Internal_ID=N_PR_7150_002D_&page_name=Chapter3,requirement,SET/SRS,Can be checked at the document level by determining whether the SRS specifies applicable reusability requirements.,NaN
2,npr-3.11.8-requirement,SWE-210,3.11.8,"The project manager shall identify software requirements for the collection, reporting, and storage of data relating...",NPR 7150.2D,https://nodis3.gsfc.nasa.gov/displayDir.cfm?Internal_ID=N_PR_7150_002D_&page_name=Chapter3,requirement,SET/SRS,"Can be checked at the SRS level by determining whether requirements exist for collecting, reporting, and storing dat...",NaN


## 2. Retrieval – TF-IDF + cosine similarity

Every requirement and every clause is turned into a vector of word scores:

- **TF** (term frequency): how often a word appears in *this* text.
- **IDF** (inverse document frequency): how rare the word is across *all* 116 texts.
  Words like "shall" appear everywhere → score ≈ 0. Words like "fault" are rare → high score.
- **TF-IDF = TF × IDF**, so each text is described by its distinctive words.

**Cosine similarity** measures the angle between two vectors: 1.0 = same words, 0.0 = nothing in common.
We compute it for all 54 × 62 pairs and keep the 3 most similar clauses per requirement.
These become the `cited_chunk_ids`. They are **not** used to decide the label.

In [3]:
# Fit the vocabulary on BOTH tables so requirements and clauses share the same word space.
vec = TfidfVectorizer(stop_words="english")          # drop "the", "shall", "of", ...
vec.fit(pd.concat([reqs["text"], clauses["text"]]))

R = vec.transform(reqs["text"])      # 54 requirement vectors, one number per vocabulary word
Cm = vec.transform(clauses["text"])  # 62 clause vectors
sim = cosine_similarity(R, Cm)       # 54 × 62 grid; sim[i, j] = how similar requirement i is to clause j (0–1)

TOP_K = 3                            # how many clauses to cite per requirement
print("similarity matrix:", sim.shape)
print("vocabulary size:  ", len(vec.vocabulary_))

# Quick look: best clause for the first requirement
i = 0                                # row 0 = SFMC-REQ-001
j = sim[i].argmax()                  # position of the highest score in that row, e.g. 17
print()
print(reqs.loc[i, "req_id"], "->", clauses.loc[j, "chunk_id"], f"(score {sim[i, j]:.2f})")
print("REQ:   ", reqs.loc[i, "text"])
print("CLAUSE:", clauses.loc[j, "text"][:200], "...")

similarity matrix: (54, 62)
vocabulary size:   2460

SFMC-REQ-001 -> swehb-3.2.1.2-guidance (score 0.08)
REQ:    The SFMC shall display the spacecraft's current position to the pilot.
CLAUSE: State and Mode Requirements If the software is required to operate in more than one state or mode, having requirements that distinguish each state or mode is a necessity. Identify and define each stat ...


In [4]:
# Full 54 × 62 similarity grid: rows = requirements, columns = clauses, cell = cosine score (0–1).

pd.DataFrame(sim, index=reqs["req_id"], columns=clauses["chunk_id"]).round(2).iloc[:5,:5]

chunk_id,npr-3.7.3-requirement,npr-3.10.1-requirement,npr-3.11.8-requirement,npr-3.11.8-note,npr-4.1.2-note
req_id,,,,,
SFMC-REQ-001,0.00,0.01,0.02,0.0,0.0
SFMC-REQ-002,0.00,0.01,0.01,0.0,0.0
SFMC-REQ-003,0.00,0.01,0.01,0.0,0.0
SFMC-REQ-004,0.07,0.01,0.01,0.0,0.0
SFMC-REQ-005,0.00,0.01,0.01,0.0,0.0


## 3. Judgment – keyword rules

`output_contract_v1.md` defines the **issue codes** but gives no detection method.
`requirement_quality_taxonomy.md` gives, for each defect category, a checklist of trigger words.
We map the Level A categories in `requirement_quality_taxonomy.md` onto the codes in `output_contract_schema_v1.json` and copy the trigger words into Python lists:

| Taxonomy category | Output-contract code | Line in `requirement_quality_taxonomy.md` |
|---|---|---|
| A1 Unquantified Qualifier | `VAGUE_CRITERION` | 24, plus examples on 19 |
| A2 Undefined Term | `UNDEFINED_TERM` | `Explanation` column of `manual_requirement_review.csv` |
| A3 Unverifiable Absolute | `UNVERIFIABLE_ABSOLUTE` | 58 |
| A4 Escape Clause | `ESCAPE_CLAUSE` | 73 |
| A5 Missing Response | `UNSPECIFIED_RESPONSE` | 83 |

Rule = "if any trigger word is in the lower-cased requirement text, flag it with that code."
A requirement with **no** hit is `ACCEPTABLE`. A requirement can carry several codes.

Not covered on purpose: A6 Circular Definition, A7 Scope/Allocation, A8 Compound, and all
Level B cross-requirement conflicts (e.g. REQ-035 vs REQ-015). Those need reading two requirements
together or understanding meaning, which a word list cannot do. See §6.

In [5]:
RULES = {
    # A1 Unquantified Qualifier -> VAGUE_CRITERION
    # A degree word with no number behind it. "quickly" instead of "within 100 ms".
    "VAGUE_CRITERION": [
        "quickly", "accurate", "reasonabl", "sufficient", "noticeable", "efficient",
        "clearly", "state-of-the-art", "immediately", "securely", "adequate", "user-friendly",
        "continuously monitor",   # REQ-007: how often is "continuously"? (kept narrow so REQ-039 "operate continuously for 72 h" is not hit)
    ],
    # A2 Undefined Term -> UNDEFINED_TERM
    # A category the SRS never defines. Who decides what is "friendly" or "important"?
    "UNDEFINED_TERM": [
        "friendly", "important", "critical fault", "seriously damaged",
        "safe location", "safest", "nonessential", "dangerous",
    ],
    # A3 Unverifiable Absolute -> UNVERIFIABLE_ABSOLUTE
    # "never crash" cannot be tested in finite time.
    "UNVERIFIABLE_ABSOLUTE": ["never", "always"],
    # A4 Escape Clause -> ESCAPE_CLAUSE
    # A phrase that lets the implementer skip the requirement.
    "ESCAPE_CLAUSE": [
        "whenever possible", "where practical", "as appropriate",
        "if feasible", "to the extent feasible", "if possible",
    ],
    # A5 Missing Response -> UNSPECIFIED_RESPONSE
    # The trigger is stated, the action is not.
    "UNSPECIFIED_RESPONSE": ["appropriate corrective action", "appropriate action", "corrective action"],
}

import re

# Match at the START of a word only (\b = word boundary).
#   "reasonabl" still catches "reasonable" and "reasonably" (no boundary at the end on purpose),
#   but "never" no longer fires inside "whenever" (REQ-043 bug found on first run).
def find_hits(text: str) -> dict:
    """Return {issue_code: [matched words]} for every rule that fires on this text."""
    t = text.lower()                           # "The SFMC shall respond Quickly" -> "the sfmc shall respond quickly"
    hits = {}
    for code, words in RULES.items():          # code = "VAGUE_CRITERION", words = ["quickly", "accurate", ...]
        found = []
        for w in words:                        # check each trigger word
            if re.search(r"\b" + re.escape(w), t):   # word starts at a word boundary
                found.append(w)                # ["quickly"]
        if found:                              # skip codes with nothing matched
            hits[code] = found                 # {"VAGUE_CRITERION": ["quickly"]}
    return hits                                # {} if nothing fired

# Sanity check on one known-bad requirement
print(find_hits("The SFMC shall respond quickly to dangerous threats."))

{'VAGUE_CRITERION': ['quickly'], 'UNDEFINED_TERM': ['dangerous']}


## 4. Run the baseline on all 54 requirements

For each requirement we record what `output_contract_v1.md` asks for. Columns of `baseline_predictions.csv`:

| Column | Meaning |
|---|---|
| `req_id` | requirement ID from `srs_requirements.csv` |
| `quality_status` | `ACCEPTABLE` or `NEEDS_REVIEW` |
| `issue_codes` | why it needs review, using the codes from `output_contract_schema_v1.json` (`VAGUE_CRITERION`, …); `NONE` if acceptable. Several codes are `;`-separated |
| `matched_keywords` | the exact trigger word(s) found in the text |
| `reason` | one sentence a human can read: code + the word that caused it |
| `cited_chunk_ids` | top-3 most similar NASA clauses (`chunk_id` from `addressable_clauses.csv`) |
| `cited_scores` | their cosine-similarity scores, 0–1 |
| `method` | how the row was produced, so it can be told apart from RAG output later |

In [6]:
# One row per requirement: keyword rules give the label, similarity gives the citations.
rows = []
for i, r in reqs.iterrows():
    hits = find_hits(r["text"])                    # e.g. {"VAGUE_CRITERION": ["quickly"]}
    top  = sim[i].argsort()[::-1][:TOP_K]          # positions of the 3 most similar clauses

    if hits:                                       # at least one rule fired, e.g. {"VAGUE_CRITERION": ["quickly"]}
        status = "NEEDS_REVIEW"
        codes = list(hits.keys())                  # ["VAGUE_CRITERION"]
        keywords = []
        for ws in hits.values():                   # collect every matched word
            keywords.extend(ws)                    # ["quickly"]
        reasons = []
        for code, ws in hits.items():              # one sentence per code
            reasons.append(f"{code}: contains '{', '.join(ws)}'")
        reason = "; ".join(reasons)                # "VAGUE_CRITERION: contains 'quickly'"
    else:                                          # hits is {} -> no rule fired
        status = "ACCEPTABLE"                      # nothing found, not "proven perfect"
        codes = ["NONE"]
        keywords = []
        reason = "No taxonomy trigger word found."

    rows.append({                                  # lists joined with ";" to fit one CSV cell
        "req_id":           r["req_id"],
        "quality_status":   status,
        "issue_codes":      ";".join(codes),
        "matched_keywords": ";".join(keywords),
        "reason":           reason,
        "cited_chunk_ids":  ";".join(clauses["chunk_id"].iloc[top]),
        "cited_scores":     ";".join(f"{sim[i, j]:.2f}" for j in top),
        "method":           "tfidf_top3 + keyword_rules",
    })

pred = pd.DataFrame(rows)
pred.to_csv("../data/baseline_predictions.csv", index=False)

print(pred["quality_status"].value_counts(), "\n")
pred[["req_id", "quality_status", "issue_codes", "reason"]].head(12)

quality_status
ACCEPTABLE      34
NEEDS_REVIEW    20
Name: count, dtype: int64 



,req_id,quality_status,issue_codes,reason
0,SFMC-REQ-001,ACCEPTABLE,NONE,No taxonomy trigger word found.
1,SFMC-REQ-002,ACCEPTABLE,NONE,No taxonomy trigger word found.
2,SFMC-REQ-003,ACCEPTABLE,NONE,No taxonomy trigger word found.
3,SFMC-REQ-004,NEEDS_REVIEW,VAGUE_CRITERION,VAGUE_CRITERION: contains 'efficient'
4,SFMC-REQ-005,ACCEPTABLE,NONE,No taxonomy trigger word found.
5,SFMC-REQ-006,NEEDS_REVIEW,VAGUE_CRITERION,VAGUE_CRITERION: contains 'reasonabl'
6,SFMC-REQ-007,NEEDS_REVIEW,VAGUE_CRITERION,VAGUE_CRITERION: contains 'continuously monitor'
7,SFMC-REQ-008,ACCEPTABLE,NONE,No taxonomy trigger word found.
8,SFMC-REQ-009,ACCEPTABLE,NONE,No taxonomy trigger word found.
9,SFMC-REQ-010,NEEDS_REVIEW,UNSPECIFIED_RESPONSE,"UNSPECIFIED_RESPONSE: contains 'appropriate corrective action, corrective action'"


## 5a. Score against the manual review (54 labels)

The `Assessment` column in `manual_requirement_review.csv` uses `ACCEPTABLE` / `NEEDS REVIEW`; we
replace the space with an underscore so the spelling matches `output_contract_v1.md`. Then:

- **Agreement** – share of the 54 where baseline label == manual label (the manual review is our answer key).
- **Precision (NEEDS_REVIEW)** – of the ones we flagged, how many the manual review also flagged.
- **Recall (NEEDS_REVIEW)** – of the ones the manual review flagged, how many we caught.

How to read the confusion table printed below: **rows** = the manual label, **columns** = the baseline label.
The diagonal (top-left, bottom-right) is where both agree. Top-right = false alarms. Bottom-left = misses.

In [7]:
manual = pd.read_csv("../docs/sprint-weeks-1-2/manual_requirement_review.csv")
manual["manual_status"] = manual["Assessment"].str.strip().str.replace(" ", "_")   # "NEEDS REVIEW" -> "NEEDS_REVIEW"

# Put baseline label and manual label side by side, one row per requirement
compare = pred.merge(manual[["Requirement ID", "manual_status", "Issue Category"]],
                     left_on="req_id", right_on="Requirement ID")

agree = (compare["quality_status"] == compare["manual_status"]).mean()   # share of the 54 where both labels match
baseline_flagged = compare["quality_status"] == "NEEDS_REVIEW"           # True where the baseline flagged
manual_flagged   = compare["manual_status"]  == "NEEDS_REVIEW"           # True where the manual review flagged
precision = (baseline_flagged & manual_flagged).sum() / baseline_flagged.sum()   # of what we flagged, how much was right
recall    = (baseline_flagged & manual_flagged).sum() / manual_flagged.sum()     # of what should be flagged, how much we caught

print(f"Agreement with manual review: {agree:.2f}  ({(compare['quality_status']==compare['manual_status']).sum()}/54)")
print(f"Precision (NEEDS_REVIEW):     {precision:.2f}")
print(f"Recall    (NEEDS_REVIEW):     {recall:.2f}")
print()
print(pd.crosstab(compare["manual_status"], compare["quality_status"], rownames=["manual"], colnames=["baseline"]))

Agreement with manual review: 0.96  (52/54)
Precision (NEEDS_REVIEW):     1.00
Recall    (NEEDS_REVIEW):     0.91

baseline      ACCEPTABLE  NEEDS_REVIEW
manual                                
ACCEPTABLE            32             0
NEEDS_REVIEW           2            20


In [8]:
# Rows where the two labels differ, with the baseline's reason next to the manual category
disagree = compare[compare["quality_status"] != compare["manual_status"]]
disagree[["req_id", "manual_status", "quality_status", "Issue Category", "reason"]]

,req_id,manual_status,quality_status,Issue Category,reason
37,SFMC-REQ-038,NEEDS_REVIEW,ACCEPTABLE,Requirements conflict,No taxonomy trigger word found.
40,SFMC-REQ-041,NEEDS_REVIEW,ACCEPTABLE,Circular definition,No taxonomy trigger word found.


## 5b. Score against the calibration cases

`calibration_examples_for_teams_v1.csv` has 11 rows; 7 are `scope == REQUIREMENT` and name a
single `target_ids` (= our `req_id`). We check both the label and the issue code.
Only CALIBRATION rows are used; the HOLDOUT rows in `benchmark_inputs_for_teams_v1.csv` are never looked at.

The file lives in the `Sprint4_Team_Starter_DRAFT` folder next to the repo, so the path is relative to that layout.

Columns of the comparison table printed below:

| Column | Comes from | Meaning |
|---|---|---|
| `req_id` | both | requirement being checked |
| `expected_outcome` | calibration file | label the file expects |
| `quality_status` | baseline | label the baseline gave |
| `issue_code` | calibration file | the one reason the file expects (`NONE` if acceptable) |
| `issue_codes` | baseline | reason(s) the baseline gave |
| `label_ok` | computed | `expected_outcome == quality_status` |
| `code_ok` | computed | expected `issue_code` appears in our `issue_codes` |

In [9]:
from pathlib import Path

cal_path = Path("../../Sprint4_Team_Starter_DRAFT/calibration_examples_for_teams_v1.csv")

if cal_path.exists():
    cal = pd.read_csv(cal_path)
    cal = cal[cal["scope"] == "REQUIREMENT"].rename(columns={"target_ids": "req_id"})   # keep the 7 single-requirement rows
    compare_cal = cal.merge(pred, on="req_id")                                                    # expected vs. baseline, side by side
    compare_cal["label_ok"] = compare_cal["expected_outcome"] == compare_cal["quality_status"]                      # same label?
    # Same reason? Expected code must appear in our list, e.g. "VAGUE_CRITERION" in "VAGUE_CRITERION;UNDEFINED_TERM"
    code_ok = []
    for expected, ours in zip(compare_cal["issue_code"], compare_cal["issue_codes"]):
        code_ok.append(expected in ours.split(";"))
    compare_cal["code_ok"] = code_ok
    print(f"Label match: {compare_cal['label_ok'].sum()}/{len(compare_cal)}   Issue-code match: {compare_cal['code_ok'].sum()}/{len(compare_cal)}")
    display(compare_cal[["req_id", "expected_outcome", "quality_status", "issue_code", "issue_codes", "label_ok", "code_ok"]])
else:
    print("Calibration file not found at", cal_path.resolve())
    print("Place the Sprint4_Team_Starter_DRAFT folder next to the repo, or edit cal_path.")

Label match: 7/7   Issue-code match: 6/7


,req_id,expected_outcome,quality_status,issue_code,issue_codes,label_ok,code_ok
0,SFMC-REQ-002,ACCEPTABLE,ACCEPTABLE,NONE,NONE,True,True
1,SFMC-REQ-006,NEEDS_REVIEW,NEEDS_REVIEW,VAGUE_CRITERION,VAGUE_CRITERION,True,True
2,SFMC-REQ-010,NEEDS_REVIEW,NEEDS_REVIEW,UNSPECIFIED_RESPONSE,UNSPECIFIED_RESPONSE,True,True
3,SFMC-REQ-021,NEEDS_REVIEW,NEEDS_REVIEW,VAGUE_CRITERION,VAGUE_CRITERION;UNDEFINED_TERM,True,True
4,SFMC-REQ-035,NEEDS_REVIEW,NEEDS_REVIEW,SAFETY_INTERACTION,UNVERIFIABLE_ABSOLUTE,True,False
5,SFMC-REQ-037,NEEDS_REVIEW,NEEDS_REVIEW,VAGUE_CRITERION,VAGUE_CRITERION,True,True
6,SFMC-REQ-043,NEEDS_REVIEW,NEEDS_REVIEW,ESCAPE_CLAUSE,ESCAPE_CLAUSE,True,True


## 5c. Citation sanity check

`output_contract_v1.md` requires every cited `chunk_id` to exist in the evidence set. Trivially true here because we
only ever pick from `addressable_clauses.csv`, but the evaluation step will run the same check on the RAG output,
so we write it once now.


In [10]:
valid_ids = set(clauses["chunk_id"])       # the 62 IDs that are allowed to be cited
all_cited = []
for s in pred["cited_chunk_ids"]:          # each row holds "id1;id2;id3"
    all_cited.extend(s.split(";"))         # -> 54 × 3 = 162 IDs
bad = []
for c in all_cited:
    if c not in valid_ids:                 # an ID that does not exist = a made-up citation
        bad.append(c)
print(f"citations: {len(all_cited)}   invalid: {len(bad)}   citation accuracy: {1 - len(bad)/len(all_cited):.2f}")

citations: 162   invalid: 0   citation accuracy: 1.00


## 6. Findings

*(Numbers below are from the run on 2026-10-13; re-run the notebook to refresh.)*

**What the baseline gets right**
- Every Level A defect that shows up as a *word* is caught: vague qualifiers, escape clauses, absolutes,
  "appropriate corrective action", undefined categories like "friendly" / "important".
- Zero false alarms on the 32 manually ACCEPTABLE requirements once "continuously" is narrowed to
  "continuously monitor" (REQ-039 "operate continuously for 72 hours" is a real, testable number).
- All 7 REQUIREMENT calibration labels match.

**What it cannot do, by design**
- **REQ-038** (conflicts with REQ-037's "no noticeable delay") and **REQ-035** (conflicts with REQ-015)
  are Level B cross-requirement problems. The baseline flags REQ-035 only because of the word "always",
  and for the wrong code (`UNVERIFIABLE_ABSOLUTE` instead of the expected `SAFETY_INTERACTION`).
- **REQ-041** (circular: "recoverable fault … shall recover") needs meaning, not words.
- The word lists were tuned by looking at the 22 manually flagged rows, so the score in §5a is
  optimistic. The HOLDOUT rows in `benchmark_inputs_for_teams_v1.csv` are the honest test.
- First run matched "never" inside "whenever" (REQ-043). Fixed with a word-boundary regex; a reminder
  that even a word list needs a test.
- Retrieval scores are low (most top-1 similarities < 0.3) because NASA guidance and SRS requirements
  share few words. Citations are "best available", not "proven relevant".

**What the RAG system must beat**
- Agreement with the manual review above 52/54, recall on NEEDS_REVIEW above 20/22, and, crucially,
  a correct issue code on REQ-035 / REQ-038 / REQ-041 where the word list fails.

## 7. Next steps
- Once `data/answer_key.csv` exists with the output-contract label names, point §5a at it directly.
- Move §5a–5c into a reusable `evaluate.py` and run it on both this file and the RAG output.